# Chapter 11 — PCIe on ARM: ECAM, BARs, MSI-X

## Concept
PCIe on ARM uses ECAM (Enhanced Configuration Access Mechanism) for config space
access — memory-mapped at a base defined in ACPI MCFG or DT. BARs (Base Address
Registers) are allocated by the OS PCI subsystem. MSI-X (Extended Message
Signaled Interrupts) deliver interrupts via GICv3 ITS — each MSI write triggers
an LPI. ARM PCIe root complexes expose the ECAM window; the SMMU or passthrough
provides isolation.

## Lab Objectives
1. Query PCIe topology via QMP `query-pci`.
2. Confirm root complex and virtio-net device visible in `lspci`.
3. Verify MSI-X is enabled for virtio-net via `lspci -v`.
4. Confirm an LPI fires in `/proc/interrupts` after network traffic.

> **QEMU Fidelity Note** — Results from this lab are functionally valid on
> QEMU `virt` machine with HVF (macOS Apple Silicon). Behaviours that differ
> from real Neoverse silicon are annotated inline. See Section 6 of the
> project plan for the full gap table.


In [ ]:
import sys, os, pathlib, time
from IPython.display import display, HTML

# ── USER CONFIGURATION — edit these paths before running ──────────────────────
LABS_ROOT    = pathlib.Path.home() / "arm_qemu_labs"
SHARED_DIR   = LABS_ROOT / "shared"
DISK_IMAGE   = LABS_ROOT / "images" / "ubuntu-24.04-arm64.qcow2"
SEED_ISO     = LABS_ROOT / "images" / "seed.iso"   # cloud-init seed
FIRMWARE     = LABS_ROOT / "firmware" / "QEMU_EFI.fd"
CONSOLE_USER = "ubuntu"
CONSOLE_PASS = "arm-lab-2026"
CPU          = "cortex-a76"
RAM          = "2G"
SMP          = 1
# ─────────────────────────────────────────────────────────────────────────────

sys.path.insert(0, str(SHARED_DIR))
from qemu_launcher  import QEMULauncher
from qmp_client     import QMPClient
from serial_console import SerialConsole
from assert_lib     import (assert_true, assert_false, assert_equal,
                             assert_contains, assert_not_contains,
                             assert_qmp_ok, assert_in_range, summary, reset)
reset()

import shutil
assert shutil.which("qemu-system-aarch64"), (
    "FATAL: qemu-system-aarch64 not found in PATH. Run setup_qemu_labs.sh.")
assert FIRMWARE.exists(),   f"FATAL: Firmware not found at {FIRMWARE}"
assert DISK_IMAGE.exists(), f"FATAL: Disk image not found at {DISK_IMAGE}"
assert SEED_ISO.exists(),   f"FATAL: Seed ISO not found at {SEED_ISO}"
print("✓ Environment check passed")
print(f"  qemu-system-aarch64 : {shutil.which('qemu-system-aarch64')}")
print(f"  Firmware            : {FIRMWARE}")
print(f"  Disk image          : {DISK_IMAGE}")


In [ ]:
launcher = QEMULauncher(
    cpu=CPU, ram=RAM, smp=SMP,
    firmware=str(FIRMWARE),
    disk_image=str(DISK_IMAGE),
    seed_iso=str(SEED_ISO),
    extra_args=["-netdev", "user,id=net0", "-device", "virtio-net-pci,netdev=net0"],
)
launcher.launch()
launcher.wait_ready(timeout=15)
print(f"QEMU running — QMP port {launcher.qmp_port}, serial port {launcher.serial_port}")


In [ ]:
qmp = QMPClient(port=launcher.qmp_port)
qmp.connect(timeout=30)

sc = SerialConsole(port=launcher.serial_port)
sc.connect(timeout=30)

print("Waiting for guest boot (up to 120 s on HVF, longer on TCG)…")
sc.wait_for_boot(timeout=180)
sc.login(CONSOLE_USER, CONSOLE_PASS)
print("Guest ready.")


In [ ]:
# ── Step 1: Query PCI topology via QMP ───────────────────────────────────────
import json
pci_info = qmp.query_pci()
print(f"PCIe buses reported by QMP: {len(pci_info)}")
for bus in pci_info:
    print(f"\n  Bus {bus.get('bus')}: {len(bus.get('devices', []))} device(s)")
    for dev in bus.get("devices", [])[:8]:
        vid = dev.get("id", {}).get("vendor", "?")
        did = dev.get("id", {}).get("device", "?")
        cls = dev.get("class_info", {}).get("desc", "?")
        print(f"    {dev.get('slot',0):02}.{dev.get('function',0)}: "
              f"VID={vid:#06x} DID={did:#06x} [{cls}]")


In [ ]:
# ── Step 2: Run lspci in the guest ───────────────────────────────────────────
lspci_out = sc.run_command(
    "lspci 2>/dev/null || sudo apt-get install -yq pciutils > /dev/null 2>&1 && lspci",
    timeout=30
)
print("lspci output:\n", lspci_out)


In [ ]:
# ── Step 3: Check MSI-X capability on virtio-net ─────────────────────────────
msix_check = sc.run_command(
    "lspci -v 2>/dev/null | grep -A 20 'Virtio.*network\|1af4:' | "
    "grep -i 'MSI-X\|Capabilities\|Interrupt'",
    timeout=15
)
print("MSI-X capability lines:\n", msix_check)


In [ ]:
# ── Step 4: Check ECAM access (MCFG table) ───────────────────────────────────
mcfg_check = sc.run_command(
    "ls /sys/firmware/acpi/tables/MCFG 2>/dev/null && echo 'MCFG present' || "
    "echo 'MCFG absent'",
    timeout=10
)
print("MCFG:", mcfg_check.strip())

# PCI resource windows from /proc/iomem
pci_resources = sc.run_command(
    "grep -i 'PCI\|ECAM\|pcie' /proc/iomem || echo 'no PCI in iomem'",
    timeout=10
)
print("PCI in /proc/iomem:\n", pci_resources)


In [ ]:
# ── Step 5: Trigger network traffic and check LPI in /proc/interrupts ────────
# Ping gateway (QEMU user-net default GW is 10.0.2.2)
sc.run_command("ping -c 10 10.0.2.2 > /dev/null 2>&1", timeout=20)
time.sleep(2)

interrupts_pci = sc.run_command(
    "grep -E 'LPI|ITS|virtio|eth|pci' /proc/interrupts || echo 'no LPI/PCI lines'",
    timeout=10
)
print("PCI/LPI interrupt lines:\n", interrupts_pci)


In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
# QMP PCIe
assert_true(len(pci_info) > 0,
            "QMP query-pci returns at least one bus",
            detail=f"{len(pci_info)} bus(es)",
            action="Check -machine virt PCIe topology")

all_devices = []
for bus in pci_info:
    all_devices.extend(bus.get("devices", []))
assert_true(len(all_devices) > 0,
            "QMP query-pci shows at least one device",
            detail=f"{len(all_devices)} device(s)",
            action="Add -device virtio-net-pci to QEMU args")

# Root complex: vendor ID 0x1b36 (Red Hat / QEMU PCIe)
rc_found = any(
    d.get("id", {}).get("vendor", 0) in (0x1b36, 0x8086, 0x1022)
    for bus in pci_info for d in bus.get("devices", [])
)
assert_true(rc_found,
            "PCIe root complex or host bridge present in QMP query-pci",
            detail=str([d.get("id") for bus in pci_info
                        for d in bus.get("devices", [])])[:100],
            action="Check -machine virt PCIe configuration")

# lspci: virtio-net
assert_contains(lspci_out, r"[Vv]irtio|1af4",
                "virtio device visible in lspci output",
                action="Add -device virtio-net-pci to QEMU launch args")

# MSI-X
assert_contains(msix_check + lspci_out, r"MSI-X|MSIX|msix",
                "MSI-X capability present on virtio-net device",
                action="Check lspci -v output; MSI-X requires PCI transport")

# MCFG
assert_true("present" in mcfg_check or "PCI" in pci_resources,
            "ECAM / MCFG accessible (ACPI MCFG or PCI resource window)",
            detail=mcfg_check.strip(),
            action="Check UEFI firmware ACPI MCFG table generation")

# LPI in /proc/interrupts
assert_contains(interrupts_pci, r"\d+",
                "PCI/LPI interrupt counter lines present after traffic",
                detail=interrupts_pci[:100],
                action="Generate more traffic; verify virtio-net driver loaded")


In [ ]:
# ── Teardown: always runs even if earlier cells raised ────────────────────────
try:
    qmp.quit()
except Exception:
    pass
try:
    sc.close()
except Exception:
    pass
try:
    launcher.terminate()
except Exception:
    pass
print("Teardown complete. QEMU process terminated.")


## Lab Summary

| Assertion | Expected | Notes |
|-----------|----------|-------|
| QMP query-pci ≥ 1 bus | Yes | PCIe topology enumerated |
| PCIe devices present | ≥ 1 | Devices on bus |
| Root complex in QMP | Yes | Bridge/RC entry |
| virtio device in lspci | Yes | PCI transport |
| MSI-X capability | Present | GICv3 ITS path |
| ECAM/MCFG accessible | Yes | Config space mapped |
| LPI counters after traffic | > 0 | ITS MSI routing |

> **Fidelity**: PCIe MSI-X via GICv3 ITS is fully emulated in QEMU.
> ECAM window base address is QEMU-specific (not Neoverse board value).

## Next Steps
→ **Chapter 12**: Linux Kernel Boot Path — time the ARM64 boot phases on HVF.
